In [25]:
import numpy as np
import math
import random
from bandit_env import BanditTenArmedGaussian

In [26]:
env = BanditTenArmedGaussian()

# Now use it normally
observation = env.reset()

In [56]:
# initialize parameters

# number of rounds (iterations)
num_rounds = 20000

# Count of number of times an arm was pulled
count = np.zeros(10)

# Sum of rewards of each arm
sum_rewards = np.zeros(10)

# Q value which is the average reward
Q = np.zeros(10)

alpha = np.ones(10)
beta = np.ones(10)

In [49]:
Q

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [50]:
def epsilon_greedy(epsilon):
    
    rand = np.random.random()  
    if rand < epsilon:
        action =  env.action_space.sample()
    else:
        action = np.argmax(Q)
    
    return action

In [51]:
'''
if all a/b test started at the same time we go with this:

its like slice the higher one has more chance to be chosen

1. 🎰 Softmax Action Selection
Use when:

You want a smooth, probabilistic balance between exploration and exploitation.

You don't want to completely ignore suboptimal arms, even if they seem worse.

You need a strategy that allows more flexibility than ε-greedy (especially with non-stationary environments).

You're working with non-stationary rewards and want temperature control (tau) to adapt exploration.

Advantages:

Gives a chance to all actions.

Biases toward better actions.

Smoothly adjusts behavior as Q changes.

'''

def softmax(tau):

    total = sum([math.exp(arm/tau) for arm in Q])
    probs = [math.exp(arm/tau)/total for arm in Q] # normalized

    threshold = random.random()

    culminative = 0.0
    
    for i in range(len(probs)):

        culminative += probs[i]
        if culminative >= threshold:
            return i

    return np.argmax(probs) 
        
    


In [52]:
# UCB

'''
if our a/b test is not started at the same time and we may add more test in future we go with this:

UCB in a Dynamic / Growing Setting
When your population of arms isn't fixed — for example, if:

New arms are added dynamically (e.g., a new ad, a new product, a new strategy),

Or old arms are rarely chosen,

Then UCB will automatically give more chances to the less-tested ones, including newly added arms.
'''

def UCB(iters):
    
    ucb = np.zeros(10)
    
    #explore all the arms
    if iters < 10:
        return iters
    
    else:
        for arm in range(10):
            
            # calculate upper bound
            upper_bound = math.sqrt((2*math.log(sum(count))) / count[arm])
            
            # add upper bound to the Q valyue
            ucb[arm] = Q[arm] + upper_bound
            
        # return the arm which has maximum value
        return (np.argmax(ucb))

In [55]:
# tompson sampling

'''
tompson sampling is some how like ucb, in the terms that we have some balance between explore and exploit.

but the difference is :

UCB: “Let’s make sure we test the under-tested arms enough.” -> if we want the same number of ab test for all arms we go with this, as its force model to choose the one
that is not chosen enough

Thompson Sampling: “I’ll go with the one that seems best right now, based on what I believe.”
Exploration happens naturally, due to randomness in sampling.

Doesn’t force testing — some arms might get picked very little, unless they look promising.


'''


def thompson_sampling(alpha,beta):
    
    samples = [np.random.beta(alpha[i]+1,beta[i]+1) for i in range(10)]

    return np.argmax(samples)

In [53]:
env.reset()
for i in range(num_rounds):

    #arm = epsilon_greedy(0.3)#choose slot
    #arm = softmax(0.5)
    #arm = UCB(i)
    ar
    observation, reward, done, info = env.step(arm)
    count[arm] +=1
    sum_rewards[arm] +=reward
    Q[arm] = sum_rewards[arm]/count[arm]

print( 'The optimal arm is {}'.format(np.argmax(Q)))

The optimal arm is 8


In [54]:
Q

array([-0.14709433,  0.89266923,  0.23433485, -0.82096256, -0.05593732,
        0.97579565,  0.23387022,  1.53011133,  2.15734828, -0.10903708])